# 01 - Data exploration: Home Credit Default Risk

Goal: understand the data before building a PD model.

Issue 2

## Imports and load

In [23]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

DATA_DIR = "../data/raw"
df = pd.read_csv(f"{DATA_DIR}/application_train.csv")
print(f"Rows: {df.shape[0]:,}  Columns: {df.shape[1]}")

Rows: 307,511  Columns: 122


## Target

`TARGET = 1` means the client had payment difficulties (our "default"), `0` otherwise.

In [24]:
default_rate = df["TARGET"].mean() #average of the TARGET column, which indicates the default rate
print(f"Default rate: {default_rate:.2%}") #in percentage with 2 decimal places
df["TARGET"].value_counts()

Default rate: 8.07%


TARGET
0    282686
1     24825
Name: count, dtype: int64

## Column types

In [25]:
print(df.dtypes.value_counts())
print("\nNon-numeric columns and their number of categories:")
df.select_dtypes(exclude="number").nunique().sort_values(ascending=False)

float64    65
int64      41
object     16
Name: count, dtype: int64

Non-numeric columns and their number of categories:


ORGANIZATION_TYPE             58
OCCUPATION_TYPE               18
NAME_INCOME_TYPE               8
NAME_TYPE_SUITE                7
WEEKDAY_APPR_PROCESS_START     7
WALLSMATERIAL_MODE             7
NAME_FAMILY_STATUS             6
NAME_HOUSING_TYPE              6
NAME_EDUCATION_TYPE            5
FONDKAPREMONT_MODE             4
CODE_GENDER                    3
HOUSETYPE_MODE                 3
NAME_CONTRACT_TYPE             2
FLAG_OWN_CAR                   2
FLAG_OWN_REALTY                2
EMERGENCYSTATE_MODE            2
dtype: int64

## Missing values

In [29]:
missing = df.isna().mean().sort_values(ascending=False)
high_missing = missing[missing > 0.5]

pd.set_option("display.max_rows", None)   # show every row, not a shortened list
print(high_missing.map("{:.1%}".format))
print(f"Total columns with >50% missing: {len(high_missing)} of {df.shape[1]}")

COMMONAREA_MEDI             69.9%
COMMONAREA_AVG              69.9%
COMMONAREA_MODE             69.9%
NONLIVINGAPARTMENTS_MODE    69.4%
NONLIVINGAPARTMENTS_AVG     69.4%
NONLIVINGAPARTMENTS_MEDI    69.4%
FONDKAPREMONT_MODE          68.4%
LIVINGAPARTMENTS_MODE       68.4%
LIVINGAPARTMENTS_AVG        68.4%
LIVINGAPARTMENTS_MEDI       68.4%
FLOORSMIN_AVG               67.8%
FLOORSMIN_MODE              67.8%
FLOORSMIN_MEDI              67.8%
YEARS_BUILD_MEDI            66.5%
YEARS_BUILD_MODE            66.5%
YEARS_BUILD_AVG             66.5%
OWN_CAR_AGE                 66.0%
LANDAREA_MEDI               59.4%
LANDAREA_MODE               59.4%
LANDAREA_AVG                59.4%
BASEMENTAREA_MEDI           58.5%
BASEMENTAREA_AVG            58.5%
BASEMENTAREA_MODE           58.5%
EXT_SOURCE_1                56.4%
NONLIVINGAREA_MODE          55.2%
NONLIVINGAREA_AVG           55.2%
NONLIVINGAREA_MEDI          55.2%
ELEVATORS_MEDI              53.3%
ELEVATORS_AVG               53.3%
ELEVATORS_MODE

## Is missingness informative?

Some columns are mostly empty. Before dropping them, check whether being missing is itself linked to default.

In [30]:
for col in ["COMMONAREA_AVG", "OWN_CAR_AGE", "YEARS_BUILD_AVG"]:
    rate = df.groupby(df[col].isna())["TARGET"].mean()
    print(f"{col}: missing {rate[True]:.2%}, present {rate[False]:.2%}")

print(pd.crosstab(df["FLAG_OWN_CAR"], df["OWN_CAR_AGE"].isna()))

COMMONAREA_AVG: missing 8.57%, present 6.91%
OWN_CAR_AGE: missing 8.50%, present 7.24%
YEARS_BUILD_AVG: missing 8.68%, present 6.87%
OWN_CAR_AGE    False   True 
FLAG_OWN_CAR                
N                  0  202924
Y             104582       5


## Default rate by category

Default rate and number of clients for selected categorical variables.

In [31]:
for col in ["NAME_INCOME_TYPE", "NAME_EDUCATION_TYPE", "NAME_CONTRACT_TYPE"]:
    summary = df.groupby(col)["TARGET"].agg(["mean", "count"])
    summary["mean"] = summary["mean"].map("{:.1%}".format)
    print(summary.sort_values("count", ascending=False), "\n")

                       mean   count
NAME_INCOME_TYPE                   
Working                9.6%  158774
Commercial associate   7.5%   71617
Pensioner              5.4%   55362
State servant          5.8%   21703
Unemployed            36.4%      22
Student                0.0%      18
Businessman            0.0%      10
Maternity leave       40.0%       5 

                                mean   count
NAME_EDUCATION_TYPE                         
Secondary / secondary special   8.9%  218391
Higher education                5.4%   74863
Incomplete higher               8.5%   10277
Lower secondary                10.9%    3816
Academic degree                 1.8%     164 

                    mean   count
NAME_CONTRACT_TYPE              
Cash loans          8.3%  278232
Revolving loans     5.5%   29279 



## Cash loans vs revolving loans

Revolving loans have a lower default rate. Compare loan size, income and external score to find out why.

In [32]:
cols = ["AMT_CREDIT", "AMT_INCOME_TOTAL", "EXT_SOURCE_2"]
print(df.groupby("NAME_CONTRACT_TYPE")[cols].median())

df["CREDIT_INCOME_RATIO"] = df["AMT_CREDIT"] / df["AMT_INCOME_TOTAL"]
print(df.groupby("NAME_CONTRACT_TYPE")["CREDIT_INCOME_RATIO"].median())

                    AMT_CREDIT  AMT_INCOME_TOTAL  EXT_SOURCE_2
NAME_CONTRACT_TYPE                                            
Cash loans            540000.0          153000.0      0.565381
Revolving loans       270000.0          135000.0      0.571315
NAME_CONTRACT_TYPE
Cash loans         3.5
Revolving loans    2.0
Name: CREDIT_INCOME_RATIO, dtype: float64


## Gender

`CODE_GENDER` will be excluded from the model: using sex as a credit criterion is treated as discriminatory under EU and Swedish anti-discrimination rules.

In [33]:
df["CODE_GENDER"].value_counts()

CODE_GENDER
F      202448
M      105059
XNA         4
Name: count, dtype: int64

## Data quirk: DAYS_EMPLOYED

The value 365243 (about 1,000 years) is a placeholder, not a real number of days.

In [ ]:
anom = df["DAYS_EMPLOYED"] == 365243
print(f"Placeholder DAYS_EMPLOYED rows: {anom.sum():,} ({anom.mean():.1%})")
print(f"Default rate, placeholder rows: {df.loc[anom, 'TARGET'].mean():.2%}")
print(f"Default rate, other rows:       {df.loc[~anom, 'TARGET'].mean():.2%}")

## Default rate by age group

A first look at a risk driver. `DAYS_BIRTH` is negative (days before the application).

In [ ]:
df["AGE_YEARS"] = -df["DAYS_BIRTH"] / 365
df["AGE_BIN"] = pd.cut(df["AGE_YEARS"], bins=range(20, 75, 5))
age_dr = df.groupby("AGE_BIN", observed=True)["TARGET"].mean()

age_dr.plot(kind="bar", title="Default rate by age group", ylabel="Default rate")
plt.tight_layout()
plt.show()

## Strongest single predictors

Absolute correlation with `TARGET`, numeric columns only.

In [ ]:
corr = df.select_dtypes("number").corr()["TARGET"].drop("TARGET")
corr.abs().sort_values(ascending=False).head(15)

## Key findings

_Write your observations here after running all cells._